In [ ]:
%pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
#%%cleaning demand dataset
# 1. Load the Data
df = pd.read_excel(r"/content/PGCB_date_power_demand.xlsx")

# Convert datetime column to actual datetime objects
df['datetime'] = pd.to_datetime(df['datetime'])

# 2. Resolve Duplicate Entries
# First, remove exact duplicate rows
df = df.drop_duplicates()

# Resolve duplicate timestamps (different data for the same time) by taking the mean
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_clean = df.groupby('datetime')[numeric_cols].mean().reset_index()

# 3. Regularize Irregular Timestamp Frequencies
# Sort the data by time
df_clean = df_clean.set_index('datetime').sort_index()

# Resample to a consistent 1-hour frequency (the most common interval in the raw data)
# This handles the irregular 30-minute 'Peak' entries by averaging them into the hour
df_resampled = df_clean.resample('1H').mean()

# 4. Anomaly Detection (Mathematical Smoothing Strategy)
# We use a Rolling Hampel Filter approach:
# Using a 24-hour window to calculate local median and standard deviation
window_size = 24
threshold = 3  # Points beyond 3 standard deviations are treated as spikes

rolling_median = df_resampled['demand_mw'].rolling(window=window_size, center=True).median()
rolling_std = df_resampled['demand_mw'].rolling(window=window_size, center=True).std()

# Identify extreme spikes or undocumented physical impossibilities (like demand <= 0)
is_anomaly = (np.abs(df_resampled['demand_mw'] - rolling_median) > (threshold * rolling_std))
is_anomaly = is_anomaly | (df_resampled['demand_mw'] <= 0)

print(f"Total rows processed: {len(df_resampled)}")
print(f"Anomalies/Spikes detected and resolved: {is_anomaly.sum()}")

# 5. Resolve Anomalies and Fill Gaps
# Replace detected spikes with NaN
df_resampled.loc[is_anomaly, 'demand_mw'] = np.nan

# Implement linear interpolation to smooth the spikes and fill missing hourly gaps
df_final = df_resampled.interpolate(method='linear')

# Final check: fill any remaining edge cases (start or end of file)
df_final = df_final.bfill().ffill()

# 6. Save the Output
df_final.to_csv('cleaned_power_demand.csv')
print("The data has been cleaned and saved to 'cleaned_power_demand.csv'.")

# Preview the results
print("\nFirst 5 rows of cleaned data:")
print(df_final[['demand_mw', 'generation_mw']].head())

In [ ]:
#%%cleaningweather
import pandas as pd

# 1. Load the Demand Dataset
demand_df = pd.read_csv(r"/content/cleaned_power_demand.csv")
demand_df['datetime'] = pd.to_datetime(demand_df['datetime'])

# 2. Load the Weather Dataset (Placeholder filename)
# Replace 'weather_data.csv' with your actual weather file name
weather_df = pd.read_excel(r"/content/weather_data.xlsx",skiprows=3)
# 5. Remove the first 3 rows
# As requested, and also helps remove initial NaNs created by lagging
weather_df.reset_index(drop=True, inplace=True)
print(weather_df.head(5))
weather_df.set_index('time', inplace=True)

# 3. Resample to the hour using the mean
# This handles half-hourly or irregular recordings by averaging them into hourly bins
weather_hourly = weather_df.resample('H').mean()

# 4. Create Lagged Weather Features
# Example: Creating 1-hour and 2-hour lags for temperature
# Heating/cooling of buildings often lags behind external temperature changes
weather_hourly['temp_lag_1'] = weather_hourly['temperature_2m (°C)'].shift(1)
weather_hourly['temp_lag_2'] = weather_hourly['temperature_2m (°C)'].shift(2)


# 6. Synchronization: Match demand timestamps exactly
# We filter the weather data to only include timestamps present in the demand data
demand_timestamps = demand_df['datetime'].unique()
weather_synchronized = weather_hourly[weather_hourly.index.isin(demand_timestamps)].copy()

# Reset index if you want 'datetime' back as a column
weather_synchronized.reset_index(inplace=True)

# Display results
print("Processed Weather Data Head:")
print(weather_synchronized.head())

# Optional: Save the cleaned weather data to a new CSV
weather_synchronized.to_csv('processed_weather_data.csv', index=False)

In [ ]:
#%%cleaning economic dataset
import pandas as pd
import numpy as np

# 1. Structure and Integrity Check
# Load the raw dataset
df = pd.read_csv(r"/content/economic_full_1.csv")

# Define indicators relevant to energy consumption
indicators_to_keep = [
    'GDP growth (annual %)',
    'GDP (constant 2015 US$)',
    'Population, total',
    'Manufacturing, value added (% of GDP)',
    'Industry (including construction), value added (% of GDP)'
]

# Filter for relevant rows and drop redundant metadata columns
df_filtered = df[df['Indicator Name'].isin(indicators_to_keep)].copy()
df_filtered.drop(columns=['Indicator Code', 'Country Name'], inplace=True, errors='ignore')

# Reshaping: "Melt" the years from columns into a single 'Year' column
year_columns = [col for col in df_filtered.columns if col.isdigit()]
df_melted = df_filtered.melt(
    id_vars=['Indicator Name'],
    value_vars=year_columns,
    var_name='Year',
    value_name='Value'
)

# Convert Year to integer and pivot to get indicators as columns
df_melted['Year'] = df_melted['Year'].astype(int)
df_pivoted = df_melted.pivot(index='Year', columns='Indicator Name', values='Value')

# 2. Handling Missing Yearly Values
# Forward fill to handle lagged reporting (e.g., 2024/2025)
# and backward fill for any early missing values
df_pivoted = df_pivoted.ffill().bfill()

# 3. Granularity Transformation (Upsampling)
# Create a date range at hourly frequency for the entire year span
start_year = df_pivoted.index.min()
end_year = df_pivoted.index.max()
hourly_index = pd.date_range(
    start=f'{start_year}-01-01 00:00:00',
    end=f'{end_year}-12-31 23:00:00',
    freq='h'
)

# Create a skeleton DataFrame with the hourly index
hourly_df = pd.DataFrame(index=hourly_index)
hourly_df['Year'] = hourly_df.index.year

# The Broadcasting Method: Merge the annual values into the hourly skeleton based on Year
final_hourly_df = hourly_df.merge(
    df_pivoted,
    left_on='Year',
    right_index=True,
    how='left'
)

# Clean up by removing the helper Year column
final_hourly_df.drop(columns=['Year'], inplace=True)

# Final Review and Export
print("Transformation Complete. Preview of the hourly economic data:")
print(final_hourly_df.head())

# Save to CSV
final_hourly_df.to_csv('processed_economic_hourly.csv')

In [ ]:
#%%merging all datasets
import pandas as pd
import numpy as np

# 1. Load the cleaned/processed datasets
demand_df = pd.read_csv(r"/content/cleaned_power_demand.csv")
weather_df = pd.read_csv(r"/content/processed_weather_data.csv")
economic_df = pd.read_csv(r"/content/processed_economic_hourly.csv")

# 2. Pre-processing Timestamps
# Ensure all time columns are in datetime format for accurate merging
demand_df['datetime'] = pd.to_datetime(demand_df['datetime'])
weather_df['time'] = pd.to_datetime(weather_df['time'])

# Standardize the economic timestamp column name (handling the index column)
economic_df.rename(columns={economic_df.columns[0]: 'timestamp'}, inplace=True)
economic_df['timestamp'] = pd.to_datetime(economic_df['timestamp'])

# 3. Merging the Datasets
# Perform a Left Join using 'cleaned_power_demand' as the primary table
# First: Merge Demand with Weather
merged_df = pd.merge(
    demand_df,
    weather_df,
    left_on='datetime',
    right_on='time',
    how='left'
)

# Second: Merge the result with Economic data
merged_df = pd.merge(
    merged_df,
    economic_df,
    left_on='datetime',
    right_on='timestamp',
    how='left'
)

# Remove redundant time columns from the joined dataframes
merged_df.drop(columns=['time', 'timestamp'], inplace=True)

# 4. Final Null Check and Imputation
# In case of "Join Gaps" (missing weather/economic rows for a specific hour),
# we use forward-fill then backward-fill to maintain time-series continuity.
merged_df = merged_df.sort_values('datetime')
merged_df = merged_df.fillna(method='ffill').fillna(method='bfill')

# 2. Define high-magnitude features
# GDP and Population are typical candidates for log transformation due to their extreme scales.
features_to_log = ['GDP (constant 2015 US$)', 'Population, total']

# 3. Apply Log Transformation
# We use np.log1p (log(1+x)) which is more robust than np.log as it handles zero values gracefully.
for col in features_to_log:
    if col in merged_df.columns:
        merged_df[f'log_{col}'] = np.log1p(merged_df[col])

        # Removing the original columns to avoid multi-collinearity
        merged_df.drop(columns=[col], inplace=True)
# 4. Identify constant columns
# A column is constant if it has only 1 unique value or 0 for NaN value(already cleansed the data of NaN values)
constant_columns = [col for col in merged_df.columns if merged_df[col].nunique() <= 1 and col!='log_GDP (constant 2015 US$)']
#we keep one constant column(to prove the model does not entirely depend on the constant column for prediction)
if len(constant_columns) > 0 :
    # 3. Drop constant columns
    merged_df= merged_df.drop(columns=constant_columns)
    print(f"Successfully dropped {len(constant_columns)} columns.")

# 6. Save and Inspect
print(f"Final dataset shape: {merged_df.shape}\n {merged_df.head(10)}")
merged_df.to_csv('final_merged_data.csv', index=False)
print("Successfully saved 'final_merged_data.csv'.")

In [ ]:
#%%eda
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Load the dataset
df = pd.read_csv(r"/content/final_merged_data.csv")
df['datetime'] = pd.to_datetime(df['datetime'])

# Create the target_demand column (demand for the next hour)
df['target_demand'] = df['demand_mw'].shift(-1)
df.dropna(inplace=True)

# Extract temporal features for analysis
df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
df['year'] = df['datetime'].dt.year

# 2. Target Variable Distribution
# Checking for skewness and central tendency in the electricity demand
plt.figure(figsize=(10, 6))
plt.hist(df['target_demand'], bins=50, color='teal', edgecolor='black', alpha=0.7)
plt.title('Distribution of Target Power Demand (t+1)')
plt.xlabel('Demand (MW)')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig('eda_distribution.png')

# 3. Time Series Trends
# Resampling to daily average to visualize long-term growth and seasonal swings
df_daily = df.set_index('datetime')['target_demand'].resample('D').mean()
plt.figure(figsize=(12, 6))
plt.plot(df_daily.index, df_daily.values, color='royalblue', linewidth=1.5)
plt.title('Daily Average Power Demand (2015 - 2025)')
plt.xlabel('Date')
plt.ylabel('Average Demand (MW)')
plt.grid(True, alpha=0.3)
plt.savefig('eda_timeseries.png')

# 4. Seasonal & Hourly Patterns
# Grouping data to find typical 'peak hours' and 'peak months'
hourly_avg = df.groupby('hour')['target_demand'].mean()
monthly_avg = df.groupby('month')['target_demand'].mean()

plt.figure(figsize=(14, 5))
# Hourly subplot
plt.subplot(1, 2, 1)
plt.plot(hourly_avg.index, hourly_avg.values, marker='o', color='darkorange', linewidth=2)
plt.title('Average Demand by Hour of Day')
plt.xlabel('Hour (0-23)')
plt.ylabel('Demand (MW)')
plt.grid(True, alpha=0.3)
plt.savefig('eda_seasonalityhourly.png')
# Monthly subplot
plt.subplot(1, 2, 2)
plt.bar(monthly_avg.index, monthly_avg.values, color='forestgreen', alpha=0.8)
plt.title('Average Demand by Month of Year')
plt.xlabel('Month (1-12)')
plt.ylabel('Demand (MW)')
plt.xticks(range(1, 13))
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('eda_seasonalitymonthly.png')

# 5. Correlation Heatmap (Matplotlib implementation)
# Visualizing how environmental and economic factors correlate with demand
cols_to_corr = ['target_demand', 'demand_mw', 'temperature_2m (°C)','relative_humidity_2m (%)', 'log_GDP (constant 2015 US$)', 'log_Population, total', 'temp_lag_1', 'load_shedding']
cols_to_corr = [c for c in cols_to_corr if c in df.columns]
corr_matrix = df[cols_to_corr].corr()

plt.figure(figsize=(10, 8))
im = plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im)
plt.xticks(range(len(cols_to_corr)), cols_to_corr, rotation=45, ha='right')
plt.yticks(range(len(cols_to_corr)), cols_to_corr)
plt.title('Correlation Matrix of Key Features')

# Overlaying correlation coefficients
for i in range(len(cols_to_corr)):
    for j in range(len(cols_to_corr)):
        plt.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha='center', va='center')
plt.tight_layout()
plt.savefig('eda_correlation.png')

# 6. Temperature vs. Demand Relationship
# Visualizing the cooling-driven demand using a scatter plot
sample_df = df.sample(2000, random_state=42)
plt.figure(figsize=(10, 6))
plt.scatter(sample_df['temperature_2m (°C)'], sample_df['target_demand'], alpha=0.5, color='crimson', s=15)
plt.title('Temperature vs. Next-Hour Demand (Sampled Points)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Target Demand (MW)')
plt.grid(True, alpha=0.3)
plt.savefig('eda_scatter.png')

In [ ]:
%pip install xgboost
%pip install optuna

In [ ]:
#%%pipeline
import pandas as pd
import numpy as np
import optuna
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit

# Load and preprocess initial dataframe
df = pd.read_csv(r"/content/final_merged_data.csv")
df['datetime'] = pd.to_datetime(df['datetime'])
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

class PredictiveParadoxPipeline:
    def __init__(self, test_year):
        self.test_year = test_year
        self.best_params = {}
        self.features = []

    def clean_data(self, df):
        """Resolves duplicates and handles anomalies."""
        df = df.groupby(df.index).mean()
        rolling_median = df['demand_mw'].rolling(window=24, center=True).median()
        std = df['demand_mw'].std()
        outliers = (df['demand_mw'] - rolling_median).abs() > (3 * std)
        df.loc[outliers, 'demand_mw'] = rolling_median[outliers]
        return df

    def engineer_features(self, df):
        """Creates temporal, historical, and delta features."""
        df = df.copy()

        # 1. Target: demand at t+1
        df['target'] = df['demand_mw'].shift(-1)

        # 2. Temporal flags
        df['hour'] = df.index.hour
        df['day_of_week'] = df.index.dayofweek
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

        # 3. Lags and Delta (Rate of Change)
        for lag in [1, 2, 24]:
            df[f'lag_{lag}h'] = df['demand_mw'].shift(lag)

        # Delta: Difference between current demand and 1 hour ago
        df['demand_delta'] = df['demand_mw'] - df['demand_mw'].shift(1)

        # 4. Rolling stats
        df['rolling_mean_24h'] = df['demand_mw'].shift(1).rolling(window=24).mean()

        df = df.dropna()
        self.features = [col for col in df.columns if col not in ['target']]
        return df

    def optimize_hyperparameters(self, X, y, model_type="xgboost"):
        """Uses Optuna to find the best parameters."""
        def objective(trial):
            if model_type == "xgboost":
                params = {
                    'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
                    'max_depth': trial.suggest_int('max_depth', 3, 10),
                    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                    'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                    'n_jobs': -1
                }
                model = XGBRegressor(**params)
            else: # Random Forest
                params = {
                    'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                    'max_depth': trial.suggest_int('max_depth', 5, 30),
                    'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                    'n_jobs': -1
                }
                model = RandomForestRegressor(**params)

            # TimeSeriesSplit for validation
            tscv = TimeSeriesSplit(n_splits=3)
            mapes = []
            for train_idx, val_idx in tscv.split(X):
                X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
                y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
                model.fit(X_t, y_t)
                preds = model.predict(X_v)
                mapes.append(mean_absolute_percentage_error(y_v, preds))

            return np.mean(mapes)

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=10) # Increase n_trials for better accuracy
        return study.best_params

    def run_pipeline(self, merged_df):
        # 1. Prep Data
        df_cleaned = self.clean_data(merged_df)
        df_final = self.engineer_features(df_cleaned)

        # 2. Year Check / Split
        train = df_final[df_final.index.year < self.test_year]
        test = df_final[df_final.index.year == self.test_year]

        if test.empty:
            raise ValueError(f"No data found for test year {self.test_year}. Check dataset range.")

        X_train, y_train = train[self.features], train['target']
        X_test, y_test = test[self.features], test['target']

        results = {}
        trained_estimators = []

        # 3. Optimize and Train Models
        for m_type in ["rf", "xgboost"]:
            print(f"--- Optimizing {m_type.upper()} ---")
            best_params = self.optimize_hyperparameters(X_train, y_train, model_type=m_type)

            if m_type == "xgboost":
                model = XGBRegressor(**best_params)
            else:
                model = RandomForestRegressor(**best_params)

            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            mape = mean_absolute_percentage_error(y_test, preds) * 100

            results[m_type] = {"model": model, "mape": mape}
            trained_estimators.append((m_type, model))
            print(f"{m_type.upper()} Test MAPE: {mape:.2f}%")

        # 4. Ensemble Component
        print("--- Building ENSEMBLE ---")
        ensemble_model = VotingRegressor(estimators=trained_estimators)
        ensemble_model.fit(X_train, y_train)

        ensemble_preds = ensemble_model.predict(X_test)
        ensemble_mape = mean_absolute_percentage_error(y_test, ensemble_preds) * 100

        results["ensemble"] = {"model": ensemble_model, "mape": ensemble_mape}
        print(f"ENSEMBLE Test MAPE: {ensemble_mape:.2f}%")

        # 5. Return the best performing model
        best_model_name = min(results, key=lambda x: results[x]['mape'])
        print(f"\nPipeline Complete. Best Model: {best_model_name.upper()}")

        return results[best_model_name]['model'], self.features

# Execute
pipeline = PredictiveParadoxPipeline(test_year=2024)
best_model, final_features = pipeline.run_pipeline(df)

[I 2026-04-24 17:24:33,331] A new study created in memory with name: no-name-857e8cf7-cb03-429b-9981-f437afe5a644


--- Optimizing RF ---
